### SNP-BINN Pipeline: From Genotypes to Disease Prediction
## =======================================================
### This notebook implements a complete pipeline for predicting disease from SNP data
### using Biologically Informed Neural Networks (BINN) with Reactome pathways.
##
### Pipeline: SNPs → Genes → Pathways → Disease Prediction

### Installation and Setup
### First, install required packages

### !pip install binn scipy pandas numpy requests

In [30]:
import numpy as np
import pandas as pd
import json
from scipy import sparse
from binn import BINN, BINNDataLoader, BINNTrainer, BINNExplainer
import warnings
warnings.filterwarnings('ignore')

## 1. Data Loading Functions
## These functions handle loading and preprocessing of input data

In [31]:
def load_genotype_data(file_path='dummy_genotype_matrix.npz'):
    """
    Load SNP genotype matrix and metadata.
    
    Returns:
        genotype_matrix: (n_patients, n_snps) array
        snps: list of SNP identifiers
        samples: list of sample identifiers
    """
    print("Loading genotype data...")
    
    loaded = np.load(file_path, allow_pickle=True)
    genotype_matrix = loaded['matrix']
    snps = loaded['snps']
    samples = loaded['samples']
    
    print(f"   Genotype matrix: {genotype_matrix.shape}")
    print(f"   SNPs: {len(snps)}, Samples: {len(samples)}")
    
    return genotype_matrix, snps, samples
    
def load_snp_gene_mapping(file_path='sparse_matrix.npz'):
    """
    Load SNP-to-Gene connectivity matrix.
    This matrix defines which SNPs influence which genes.
    
    Returns:
        snp_to_gene: (n_snps, n_genes) sparse matrix
    """
    print("Loading SNP-to-Gene mapping...")
    
    sparse_loaded = np.load(file_path, allow_pickle=True)
    
    # Handle different formats
    if 'matrix' in sparse_loaded.keys():
        snp_to_gene_sparse = sparse_loaded['matrix']
    elif len(sparse_loaded.keys()) == 1:
        key = list(sparse_loaded.keys())[0]
        snp_to_gene_sparse = sparse_loaded[key]
    else:
        snp_to_gene_sparse = sparse.load_npz(file_path)
    
    print(f"   SNP-to-Gene matrix: {snp_to_gene_sparse.shape}")
    return snp_to_gene_sparse

def load_indices():
    """
    Load SNP and gene indices for mapping identifiers to matrix positions.
    
    Returns:
        snp_index: dict mapping SNP names to indices
        gene_index: dict mapping gene names to indices
    """
    print("Loading indices...")
    
    with open('snp_index.json', 'r') as f:
        snp_index = json.load(f)
    with open('gene_index.json', 'r') as f:
        gene_index = json.load(f)
    
    print(f"   SNPs: {len(snp_index)}, Genes: {len(gene_index)}")
    return snp_index, gene_index

## 2. SNP-to-Gene Transformation
## Convert SNP genotypes to gene-level signals using biological mappings

In [32]:
def transform_snps_to_genes(genotype_matrix, snp_to_gene_sparse):
    """
    Transform SNP data to gene-level signals using matrix multiplication.
    
    This step aggregates SNP effects into gene-level features based on
    biological knowledge of which SNPs affect which genes.
    
    Args:
        genotype_matrix: (n_patients, n_snps) 
        snp_to_gene_sparse: (n_snps, n_genes)
    
    Returns:
        gene_signals: (n_patients, n_genes)
    """
    print("Transforming SNPs to gene signals...")
    
    # Convert sparse to dense if needed
    if sparse.issparse(snp_to_gene_sparse):
        snp_to_gene = snp_to_gene_sparse.toarray()
    else:
        snp_to_gene = snp_to_gene_sparse
    
    # Matrix multiplication: patients × SNPs @ SNPs × genes = patients × genes
    gene_signals = genotype_matrix @ snp_to_gene
    
    print(f"Gene signals shape: {gene_signals.shape}")
    return gene_signals



## 3. Gene-to-Pathway Mapping
## Create mappings from genes to Reactome biological pathways

In [33]:
def create_gene_pathway_mapping():
    """
    Parse Reactome GMT file to create gene-to-pathway mappings.
    
    GMT format: pathway_name \t reactome_id \t gene1 \t gene2 \t ...
    
    Critical fix: Use column 1 (Reactome IDs) not column 0 (pathway names)
    for compatibility with pathway hierarchy.
    
    Returns:
        mapping_df: DataFrame with 'input' (gene) and 'translation' (pathway_id)
    """
    print("Creating gene-to-pathway mapping...")
    
    gene_to_pathway = []
    
    with open('ReactomePathways.gmt', 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 3:
                # CRITICAL: Use parts[1] (Reactome ID) not parts[0] (pathway name)
                reactome_id = parts[1]  # "R-HSA-164843"
                genes = parts[2:]       # ["BANF1", "HMGA1", ...]
                
                for gene in genes:
                    gene_to_pathway.append({
                        'input': gene,           # Gene symbol
                        'translation': reactome_id  # Reactome pathway ID
                    })
    
    mapping_df = pd.DataFrame(gene_to_pathway)
    print(f"   Created {len(mapping_df)} gene-pathway mappings")
    
    return mapping_df

def create_pathway_hierarchy():
    """
    Load pathway hierarchy and add root node for BINN compatibility.
    
    BINN expects a root node called 'output_node' connected to top-level pathways.
    
    Returns:
        pathways_df_complete: DataFrame with pathway connections
    """
    print("Setting up pathway hierarchy...")
    
    # Load pathway relationships
    pathways_df = pd.read_csv('reactome_pathways_relation_2025_01_14.txt', sep='\t')
    pathways_df.columns = ['source', 'target']
    
    # Find top-level pathways (sources that aren't targets of other pathways)
    all_targets = set(pathways_df['target'])
    all_sources = set(pathways_df['source'])
    top_level = all_sources - all_targets
    
    # Add output_node connections to top-level pathways
    output_connections = pd.DataFrame({
        'source': ['output_node'] * len(top_level),
        'target': list(top_level)
    })
    
    pathways_df_complete = pd.concat([pathways_df, output_connections], ignore_index=True)
    print(f"   Pathway hierarchy: {len(pathways_df_complete)} connections")
    
    return pathways_df_complete

## 4. Data Validation
## Verify all mappings work correctly before training

In [40]:
def validate_mappings(genotype_matrix, gene_signals, gene_names, mapping_df, pathways_df_complete):
    """
    Validate each step of the data transformation pipeline.
    
    This ensures data flows correctly through:
    SNPs → Genes → Pathways → Network
    """
    print("Validating data mappings...")
    
    # 1. SNP-to-Gene transformation
    print(f"SNP matrix: {genotype_matrix.shape}")
    print(f"Gene signals: {gene_signals.shape}")
    assert gene_signals.shape[1] == len(gene_names), "Gene count mismatch"
    
    # 2. Gene-to-Pathway coverage
    genes_in_data = set(gene_names)
    genes_in_gmt = set(mapping_df['input'])
    overlap = genes_in_data & genes_in_gmt
    print(f"   Gene coverage: {len(overlap)}/{len(genes_in_data)} genes have pathway mappings")
    
    # 3. Pathway hierarchy consistency
    pathway_ids_in_mapping = set(mapping_df['translation'])
    pathway_ids_in_hierarchy = set(pathways_df_complete['source']) | set(pathways_df_complete['target'])
    hierarchy_overlap = pathway_ids_in_mapping & pathway_ids_in_hierarchy
    print(f"   Pathway consistency: {len(hierarchy_overlap)}/{len(pathway_ids_in_mapping)} pathways connected")
    
    # 4. Sample gene validation
    sample_genes = ['VAV3', 'SDF4'] if 'VAV3' in genes_in_gmt else list(genes_in_gmt)[:2]
    for gene in sample_genes[:2]:
        pathways = mapping_df[mapping_df['input'] == gene]['translation'].tolist()
        print(f"   {gene} → {len(pathways)} pathways (e.g., {pathways[:2]})")
    
    print("All validations passed!")

## 5. BINN Model Creation
## Build the biologically informed neural network

In [41]:
def create_binn_model(gene_data, mapping_df, pathways_df_complete):
    """
    Create BINN model with gene data and pathway structure.
    
    This is where the magic happens - BINN builds a sparse neural network
    where connections follow biological pathway hierarchies.
    
    Args:
        gene_data: DataFrame with genes as features
        mapping_df: Gene-to-pathway mappings
        pathways_df_complete: Pathway hierarchy with output_node
    
    Returns:
        binn: Trained BINN model
    """
    print("Creating BINN model...")
    
    binn = BINN(
        data_matrix=gene_data,
        mapping=mapping_df,
        pathways=pathways_df_complete,
        n_layers=4,          # Number of pathway hierarchy levels
        n_outputs=2,         # Binary classification (case vs control)
        dropout=0.2          # Regularization
    )
    
    print(f"BINN created with {len(binn.connectivity_matrices)} layers")
    return binn


## 6. Training Setup
## Prepare data for model training

In [42]:
def prepare_training_data(gene_signals, gene_names):
    """
    Format gene data for BINN training.
    
    BINN expects specific DataFrame format with genes as rows.
    """
    print("Preparing training data...")
    
    sample_names = [f"Patient_{i}" for i in range(gene_signals.shape[0])]
    
    # Create DataFrame: genes as rows, patients as columns
    gene_data = pd.DataFrame(
        gene_signals.T,  # Transpose for BINN format
        index=gene_names,
        columns=sample_names
    ).reset_index()
    
    # BINN expects 'Protein' column name (legacy from proteomics origins)
    gene_data = gene_data.rename(columns={'index': 'Protein'})
    
    print(f"   Gene data matrix: {gene_data.shape}")
    return gene_data, sample_names

def create_design_matrix(sample_names, labels=None):
    """
    Create design matrix with case/control labels.
    
    Args:
        sample_names: List of sample identifiers
        labels: Optional list of actual labels (if None, creates random labels)
    
    Returns:
        design_matrix: DataFrame with sample labels
    """
    print("Creating design matrix...")
    
    if labels is None:
        # Create balanced random labels for demonstration
        n_samples = len(sample_names)
        n_case = n_samples // 2
        labels = ['case'] * n_case + ['control'] * (n_samples - n_case)
        np.random.shuffle(labels)
    
    design_matrix = pd.DataFrame({
        'sample': sample_names,
        'group': labels
    })
    
    print(f"   Labels: {pd.Series(labels).value_counts().to_dict()}")
    return design_matrix

## 7. Model Training
## Train the BINN model on gene-pathway data

In [43]:
def train_binn_model(binn, gene_data, design_matrix, epochs=50, batch_size=8):
    """
    Train BINN model for disease classification.
    
    The model learns which pathways are important for distinguishing
    between cases and controls.
    """
    print("Training BINN model...")
    
    # Create data loaders
    binn_dataloader = BINNDataLoader(binn)
    dataloaders = binn_dataloader.create_dataloaders(
        data_matrix=gene_data,
        design_matrix=design_matrix,
        feature_column="Protein",
        group_column="group",
        sample_column="sample",
        batch_size=batch_size,
        validation_split=0.2
    )
    
    # Train model
    trainer = BINNTrainer(binn)
    trainer.fit(dataloaders=dataloaders, num_epochs=epochs)
    
    print("Training complete!")
    return trainer, dataloaders

## 8. Model Interpretation
## Extract biological insights from trained model

In [44]:
def interpret_model(binn, dataloaders):
    """
    Use SHAP to interpret which pathways the model finds important.
    
    This reveals biological mechanisms underlying disease prediction.
    """
    print("Interpreting model predictions...")
    
    explainer = BINNExplainer(binn)
    explanations = explainer.explain_single(
        dataloaders, 
        split="val", 
        normalization_method="subgraph"
    )
    
    print("Model interpretation complete!")
    return explanations



def run_snp_binn_pipeline(epochs=10, batch_size=8):
    """
    Run the complete SNP-BINN pipeline.
    
    This function orchestrates all steps from data loading to model interpretation.
    """
    print("Starting SNP-BINN Pipeline")
    print("=" * 50)
    
    # Step 1: Load data
    genotype_matrix, snps, samples = load_genotype_data()
    snp_to_gene_sparse = load_snp_gene_mapping()
    snp_index, gene_index = load_indices()
    
    # Step 2: Transform SNPs to genes
    gene_signals = transform_snps_to_genes(genotype_matrix, snp_to_gene_sparse)
    gene_names = list(gene_index.keys())
    
    # Step 3: Create pathway mappings
    mapping_df = create_gene_pathway_mapping()
    pathways_df_complete = create_pathway_hierarchy()
    
    # Step 4: Validate mappings
    validate_mappings(genotype_matrix, gene_signals, gene_names, mapping_df, pathways_df_complete)
    
    # Step 5: Prepare training data
    gene_data, sample_names = prepare_training_data(gene_signals, gene_names)
    design_matrix = create_design_matrix(sample_names)
    
    # Step 6: Create and train BINN
    binn = create_binn_model(gene_data, mapping_df, pathways_df_complete)
    trainer, dataloaders = train_binn_model(binn, gene_data, design_matrix, epochs, batch_size)
    
    # Step 7: Interpret results
    explanations = interpret_model(binn, dataloaders)
    
    print("=" * 50)
    print("Pipeline Complete!")
    print("Your SNP→Gene→Pathway→Disease prediction model is ready!")
    
    return {
        'binn': binn,
        'trainer': trainer,
        'dataloaders': dataloaders,
        'explanations': explanations,
        'gene_data': gene_data,
        'design_matrix': design_matrix
    }

## 10. Run the Pipeline
## Execute the complete analysis

In [45]:
if __name__ == "__main__":
    # Run with default parameters
    results = run_snp_binn_pipeline(epochs=10, batch_size=8)
    
    # Access results
    binn_model = results['binn']
    explanations = results['explanations']
    
    print("\nFinal Results:")
    print(f"Model layers: {len(binn_model.connectivity_matrices)}")

Starting SNP-BINN Pipeline
Loading genotype data...
   Genotype matrix: (50, 100)
   SNPs: 100, Samples: 100
Loading SNP-to-Gene mapping...
   SNP-to-Gene matrix: (100, 122)
Loading indices...
   SNPs: 100, Genes: 122
Transforming SNPs to gene signals...
Gene signals shape: (50, 122)
Creating gene-to-pathway mapping...
   Created 135371 gene-pathway mappings
Setting up pathway hierarchy...
   Pathway hierarchy: 23152 connections
Validating data mappings...
SNP matrix: (50, 100)
Gene signals: (50, 122)
   Gene coverage: 51/122 genes have pathway mappings
   Pathway consistency: 2726/2785 pathways connected
   VAV3 → 46 pathways (e.g., ['R-HSA-422475', 'R-HSA-9748787'])
   SDF4 → 0 pathways (e.g., [])
All validations passed!
Preparing training data...
   Gene data matrix: (122, 51)
Creating design matrix...
   Labels: {'control': 25, 'case': 25}
Creating BINN model...

[INFO] BINN is on device: cpu
BINN created with 5 layers
Training BINN model...
Mapping group labels: {'case': 0, 'contr